# query

> The structured typed query expression (pass-2 Thread 5) — DATA nouns describing graph reads. **Execution lives in graph-storage adapters** (stage 4 translates expressions per-backend); this library only defines, validates, and (de)serializes them. Typed expressions are the primary, portable surface — no storage-schema leak (the raw-SQL `nodes`/`edges` + `json_extract` coupling of ledger C2/C3), and scale-shaped (server-side filter/page/count answering D13). `RawQuery` is the explicitly-marked, backend-coupled escape hatch: recurring raw patterns EXPOSE missing typed-expression capabilities and get PROMOTED into the typed surface (the real-world-testing forcing function).

In [ ]:
#| default_exp query

## Design notes

Replaces the old `GraphQuery` raw-string DTO. The expression power built now is what the cores' evidence demands — label + property filtering (C2/C3), batch-by-id (C17), relation traversal (corrections joins), source matching (`find_prior_corrections_by_hash`), order/page/count (D13 server-side aggregates) — while the shape admits a fuller query language later (generalize-seam-one-level).

**Operator vocabulary is reserved up front** (`PREDICATE_OPS`); executors implement progressively and MUST raise on unsupported operators, never silently ignore (reserve-enum-values-up-front discipline).

**Stage-4 promotions (2026-06-10, all evidence-backed; ledger `stage-4-evidence.md`):**

- **Dotted property paths** in `prop` / `order_by` / `project` descend nested property JSON (`payload.document_id`) — sites C-8/C-9.
- **`RelationPredicate` far-end constraints**: `node_ids` (batch far end, C17) and `node_source` (far-end provenance match — the two-hop `find_prior_corrections_by_hash` read, promoted over raw because it is the hottest per-item review read).
- **`EdgeQuery` endpoint constraints**: `source_ids`/`target_ids` (superseded-set read) and `source_related`/`target_related` (the D13 NEXT-chain count: "NEXT edges whose source is PART_OF doc"). A domain-specific `verify_spine` op is OFF the table — the adapter stays domain-neutral; verify composes from these neutral aggregates.
- **`contains` has DEFINED case-INSENSITIVE semantics** (P12: the only consumer wanted insensitive; SQLite `LIKE` vs postgres `LIKE` diverge, so semantics must be defined by the expression, never inherited from the backend). A case-sensitive variant waits for a consumer.
- **Typed result DTOs** (`NodeQueryResult`/`EdgeQueryResult`/`RawQueryResult`) mirror the query types and are wire-registered; projected rows always carry structural fields (`id`; plus `source_id`/`target_id` for edges); `sources` is projectable on nodes.

Deferred until adopter evidence: group-by aggregates, depth-N traversal expressions (whole-neighborhood reads stay on `get_context`), compound boolean predicates (AND is the implicit combination of `where`; OR/NOT wait for a real consumer — the evidenced OR case, empty-or-null text, is two server-side-filtered queries unioned client-side).

In [ ]:
#| export
from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional

from cjm_substrate.core.errors import CapabilityInputError
from cjm_substrate.core.wire import wire_type

from cjm_context_graph_primitives.locators import ResourceLocator, locator_from_dict
from cjm_context_graph_primitives.graph import GraphNode, GraphEdge

In [ ]:
#| export
PREDICATE_OPS = (  # Reserved operator vocabulary (executors may implement a subset but MUST raise on unsupported ops)
    "eq",        # property == value
    "ne",        # property != value
    "lt",        # property < value
    "le",        # property <= value
    "gt",        # property > value
    "ge",        # property >= value
    "in",        # property in value (list)
    "contains",  # value is a substring of property — DEFINED case-INSENSITIVE (P12; semantics owned by the expression, never inherited from backend collation)
    "is_null",   # property is null / absent (value must be None)
    "not_null",  # property is present and non-null (value must be None)
)

In [ ]:
#| export
@dataclass(frozen=True)
class PropertyPredicate:
    """One property comparison; a query's `where` list combines predicates with AND.

    `prop` may be a dotted path descending nested property JSON
    (e.g. `payload.document_id` — stage-4 promotion, sites C-8/C-9)."""
    prop: str          # Property name inside the entity's properties payload (dotted path descends nesting)
    op: str            # Operator from PREDICATE_OPS
    value: Any = None  # Comparison value (must be None for is_null/not_null)

    def __post_init__(self):
        if self.op not in PREDICATE_OPS:
            raise CapabilityInputError(
                f"Unknown predicate op {self.op!r} (known: {PREDICATE_OPS})",
                fields_invalid=["op"],
            )
        if self.op in ("is_null", "not_null") and self.value is not None:
            raise CapabilityInputError(
                f"Op {self.op!r} takes no value (got {self.value!r})",
                fields_invalid=["value"],
            )

    def to_dict(self) -> Dict[str, Any]:  # Wire dict
        """Serialize to the wire dict form."""
        return {"prop": self.prop, "op": self.op, "value": self.value}

    @classmethod
    def from_dict(
        cls,
        d: Dict[str, Any]  # Wire dict
    ) -> "PropertyPredicate":  # Reconstructed predicate
        """Reconstruct from the wire dict form."""
        return cls(prop=d["prop"], op=d["op"], value=d.get("value"))

In [ ]:
#| export
@dataclass(frozen=True)
class SourcePredicate:
    """Match nodes whose `sources` contain a reference matching by content hash
    and/or locator — the typed form of the reverse provenance index
    (`find_prior_corrections_by_hash`-style reads; identity-first per CR-19)."""
    content_hash: Optional[str] = None        # Match refs with this content hash (PRIMARY identity)
    locator: Optional[ResourceLocator] = None  # Match refs with this locator

    def __post_init__(self):
        if self.content_hash is None and self.locator is None:
            raise CapabilityInputError(
                "SourcePredicate requires content_hash and/or locator",
                fields_invalid=["content_hash", "locator"],
            )

    def to_dict(self) -> Dict[str, Any]:  # Wire dict
        """Serialize to the wire dict form."""
        return {
            "content_hash": self.content_hash,
            "locator": self.locator.to_dict() if self.locator is not None else None,
        }

    @classmethod
    def from_dict(
        cls,
        d: Dict[str, Any]  # Wire dict
    ) -> "SourcePredicate":  # Reconstructed predicate
        """Reconstruct from the wire dict form."""
        loc = d.get("locator")
        return cls(
            content_hash=d.get("content_hash"),
            locator=locator_from_dict(loc) if loc is not None else None,
        )

In [ ]:
#| export
RELATION_DIRECTIONS = ("out", "in", "both")  # Edge direction relative to the candidate node


@dataclass(frozen=True)
class RelationPredicate:
    """Match nodes that have an edge of `relation_type` (one-hop, typed traversal —
    e.g. "Segments PART_OF document D"). Depth-N neighborhood reads stay on
    `get_context`; richer traversal expressions wait for adopter evidence.

    Far-end constraints (stage-4 promotions, exactly one hop deep):
    `node_id`/`node_ids` pin the far-end node (batch form = C17); `node_source`
    matches the far end by provenance (the two-hop
    `find_prior_corrections_by_hash` read: "Corrections whose CORRECTS target
    carries this content hash")."""
    relation_type: str             # Edge relation type, e.g. "PART_OF", "CORRECTS"
    direction: str = "out"         # Edge direction from the candidate node: out / in / both
    node_id: Optional[str] = None  # Constrain the far end to this node id (None = any)
    node_ids: Optional[List[str]] = None          # Constrain the far end to any of these ids (batch; C17)
    node_source: Optional["SourcePredicate"] = None  # Constrain the far end by provenance match

    def __post_init__(self):
        if self.direction not in RELATION_DIRECTIONS:
            raise CapabilityInputError(
                f"Unknown direction {self.direction!r} (known: {RELATION_DIRECTIONS})",
                fields_invalid=["direction"],
            )
        if self.node_id is not None and self.node_ids is not None:
            raise CapabilityInputError(
                "Use node_id OR node_ids, not both",
                fields_invalid=["node_id", "node_ids"],
            )

    def to_dict(self) -> Dict[str, Any]:  # Wire dict
        """Serialize to the wire dict form."""
        return {"relation_type": self.relation_type, "direction": self.direction,
                "node_id": self.node_id, "node_ids": self.node_ids,
                "node_source": self.node_source.to_dict() if self.node_source is not None else None}

    @classmethod
    def from_dict(
        cls,
        d: Dict[str, Any]  # Wire dict
    ) -> "RelationPredicate":  # Reconstructed predicate
        """Reconstruct from the wire dict form."""
        ns = d.get("node_source")
        return cls(relation_type=d["relation_type"], direction=d.get("direction", "out"),
                   node_id=d.get("node_id"), node_ids=d.get("node_ids"),
                   node_source=SourcePredicate.from_dict(ns) if ns is not None else None)

In [ ]:
#| export
@dataclass(frozen=True)
class OrderBy:
    """Result ordering by one property (server-side; C2/C3 ORDER BY index)."""
    prop: str                 # Property name to order by
    descending: bool = False  # Descending order when True

    def to_dict(self) -> Dict[str, Any]:  # Wire dict
        """Serialize to the wire dict form."""
        return {"prop": self.prop, "descending": self.descending}

    @classmethod
    def from_dict(
        cls,
        d: Dict[str, Any]  # Wire dict
    ) -> "OrderBy":  # Reconstructed ordering
        """Reconstruct from the wire dict form."""
        return cls(prop=d["prop"], descending=d.get("descending", False))

In [ ]:
#| export
@dataclass
class NodeQuery:
    """Typed node read: filter / traverse / order / page / project / count.

    All filter fields combine with AND. `count=True` returns a count instead of
    rows (the D13 verify-spine aggregate shape). `project` limits returned
    properties (server-side projection; None = whole nodes). Projected rows
    ALWAYS carry the structural field `id`; the pseudo-field `"sources"` is
    projectable (the C-2 spine read needs id + properties + sources).
    """
    ids: Optional[List[str]] = None              # Batch-by-id (C17); None = no id filter
    label: Optional[str] = None                  # Node label filter
    where: List[PropertyPredicate] = field(default_factory=list)  # Property predicates (AND)
    source: Optional[SourcePredicate] = None     # Provenance reverse-index match
    related: Optional[RelationPredicate] = None  # One-hop relation constraint
    order_by: Optional[OrderBy] = None           # Server-side ordering
    limit: Optional[int] = None                  # Page size; None = backend default
    offset: int = 0                              # Page offset
    count: bool = False                          # Return count instead of rows
    project: Optional[List[str]] = None          # Property names (dotted paths ok) + "sources"; None = whole nodes

    TYPE = "node_query"  # Wire-dict type tag

    def to_dict(self) -> Dict[str, Any]:  # Tagged wire dict
        """Serialize to the wire dict form."""
        return {
            "type": self.TYPE,
            "ids": self.ids,
            "label": self.label,
            "where": [p.to_dict() for p in self.where],
            "source": self.source.to_dict() if self.source is not None else None,
            "related": self.related.to_dict() if self.related is not None else None,
            "order_by": self.order_by.to_dict() if self.order_by is not None else None,
            "limit": self.limit,
            "offset": self.offset,
            "count": self.count,
            "project": self.project,
        }

    @classmethod
    def from_dict(
        cls,
        d: Dict[str, Any]  # Tagged wire dict
    ) -> "NodeQuery":  # Reconstructed query
        """Reconstruct from the wire dict form."""
        src, rel, ob = d.get("source"), d.get("related"), d.get("order_by")
        return cls(
            ids=d.get("ids"),
            label=d.get("label"),
            where=[PropertyPredicate.from_dict(p) for p in d.get("where", [])],
            source=SourcePredicate.from_dict(src) if src is not None else None,
            related=RelationPredicate.from_dict(rel) if rel is not None else None,
            order_by=OrderBy.from_dict(ob) if ob is not None else None,
            limit=d.get("limit"),
            offset=d.get("offset", 0),
            count=d.get("count", False),
            project=d.get("project"),
        )

In [ ]:
#| export
@dataclass
class EdgeQuery:
    """Typed edge read: filter / order / page / project / count.

    Covers the cores' edge reads: counting edges by relation type for a spine
    (D13 verify aggregates) and reading edge properties off a node's edges
    (correction decisions). Projected rows ALWAYS carry the structural fields
    `id`, `source_id`, `target_id` (the review-markers read needs target_id +
    one property).

    Endpoint constraints (stage-4 promotions): `source_ids`/`target_ids` pin an
    endpoint to an id set (superseded-set read); `source_related`/
    `target_related` constrain an endpoint by ITS relations — the D13
    NEXT-chain count ("NEXT edges whose source node is PART_OF doc D") without
    materializing the document's segment ids. Exactly one hop deep, mirroring
    `RelationPredicate`'s far-end constraints.
    """
    ids: Optional[List[str]] = None       # Batch-by-id; None = no id filter
    relation_type: Optional[str] = None   # Edge relation-type filter
    source_id: Optional[str] = None       # Origin node filter (single)
    target_id: Optional[str] = None       # Destination node filter (single)
    source_ids: Optional[List[str]] = None  # Origin node filter (batch)
    target_ids: Optional[List[str]] = None  # Destination node filter (batch)
    source_related: Optional[RelationPredicate] = None  # Constrain the origin node by its relations
    target_related: Optional[RelationPredicate] = None  # Constrain the destination node by its relations
    where: List[PropertyPredicate] = field(default_factory=list)  # Property predicates (AND)
    order_by: Optional[OrderBy] = None    # Server-side ordering
    limit: Optional[int] = None           # Page size; None = backend default
    offset: int = 0                       # Page offset
    count: bool = False                   # Return count instead of rows
    project: Optional[List[str]] = None   # Property names (dotted paths ok); None = whole edges

    TYPE = "edge_query"  # Wire-dict type tag

    def __post_init__(self):
        if self.source_id is not None and self.source_ids is not None:
            raise CapabilityInputError(
                "Use source_id OR source_ids, not both",
                fields_invalid=["source_id", "source_ids"],
            )
        if self.target_id is not None and self.target_ids is not None:
            raise CapabilityInputError(
                "Use target_id OR target_ids, not both",
                fields_invalid=["target_id", "target_ids"],
            )

    def to_dict(self) -> Dict[str, Any]:  # Tagged wire dict
        """Serialize to the wire dict form."""
        return {
            "type": self.TYPE,
            "ids": self.ids,
            "relation_type": self.relation_type,
            "source_id": self.source_id,
            "target_id": self.target_id,
            "source_ids": self.source_ids,
            "target_ids": self.target_ids,
            "source_related": self.source_related.to_dict() if self.source_related is not None else None,
            "target_related": self.target_related.to_dict() if self.target_related is not None else None,
            "where": [p.to_dict() for p in self.where],
            "order_by": self.order_by.to_dict() if self.order_by is not None else None,
            "limit": self.limit,
            "offset": self.offset,
            "count": self.count,
            "project": self.project,
        }

    @classmethod
    def from_dict(
        cls,
        d: Dict[str, Any]  # Tagged wire dict
    ) -> "EdgeQuery":  # Reconstructed query
        """Reconstruct from the wire dict form."""
        ob, sr, tr = d.get("order_by"), d.get("source_related"), d.get("target_related")
        return cls(
            ids=d.get("ids"),
            relation_type=d.get("relation_type"),
            source_id=d.get("source_id"),
            target_id=d.get("target_id"),
            source_ids=d.get("source_ids"),
            target_ids=d.get("target_ids"),
            source_related=RelationPredicate.from_dict(sr) if sr is not None else None,
            target_related=RelationPredicate.from_dict(tr) if tr is not None else None,
            where=[PropertyPredicate.from_dict(p) for p in d.get("where", [])],
            order_by=OrderBy.from_dict(ob) if ob is not None else None,
            limit=d.get("limit"),
            offset=d.get("offset", 0),
            count=d.get("count", False),
            project=d.get("project"),
        )

In [ ]:
#| export
@dataclass
class RawQuery:
    """The explicitly-marked, backend-coupled escape hatch.

    For reads the typed expressions cannot yet express. `backend` is REQUIRED —
    a raw query is non-portable by construction and an executor must refuse a
    backend mismatch. Recurring raw patterns are promotion candidates into the
    typed surface (record them; that is the forcing function).
    """
    text: str      # Backend-native query text (e.g. SQL)
    backend: str   # Backend this query is coupled to (e.g. "sqlite"); executors refuse mismatches
    params: Any = field(default_factory=list)  # Positional list or named dict of parameters

    TYPE = "raw_query"  # Wire-dict type tag

    def __post_init__(self):
        if not self.backend:
            raise CapabilityInputError(
                "RawQuery requires an explicit backend (it is non-portable by construction)",
                fields_invalid=["backend"],
            )

    def to_dict(self) -> Dict[str, Any]:  # Tagged wire dict
        """Serialize to the wire dict form."""
        return {"type": self.TYPE, "text": self.text, "backend": self.backend,
                "params": self.params}

    @classmethod
    def from_dict(
        cls,
        d: Dict[str, Any]  # Tagged wire dict
    ) -> "RawQuery":  # Reconstructed query
        """Reconstruct from the wire dict form."""
        return cls(text=d["text"], backend=d["backend"], params=d.get("params", []))

In [ ]:
#| export
QUERY_TYPES: Dict[str, type] = {  # Tagged-dict registry for wire dispatch
    NodeQuery.TYPE: NodeQuery,
    EdgeQuery.TYPE: EdgeQuery,
    RawQuery.TYPE: RawQuery,
}


def query_from_dict(
    d: Dict[str, Any]  # Tagged wire dict ("type" in QUERY_TYPES)
) -> Any:  # NodeQuery | EdgeQuery | RawQuery
    """Reconstruct a query expression from its tagged wire dict."""
    t = d.get("type")
    cls = QUERY_TYPES.get(t)
    if cls is None:
        raise CapabilityInputError(
            f"Unknown query type {t!r} (known: {sorted(QUERY_TYPES)})",
            fields_invalid=["type"],
        )
    return cls.from_dict(d)

## Typed query results

Results mirror the query types (stage-4): exactly one field is populated, matching the query's mode (default / `project` / `count`). All three are **wire-registered**, so adapter query methods return them typed across the worker boundary. Projected rows always carry structural fields (`id`; plus `source_id`/`target_id` for edge rows) — the C-2 spine read and the review-markers read both select structure alongside properties.

In [ ]:
#| export
@wire_type("graph.node_query_result")
@dataclass
class NodeQueryResult:
    """Typed result of a `NodeQuery` — exactly one field is populated,
    mirroring the query's mode (default → `nodes`, project → `rows`,
    count → `count`)."""
    nodes: Optional[List[GraphNode]] = None      # Full nodes (default mode)
    rows: Optional[List[Dict[str, Any]]] = None  # Projected rows (project mode); always carry "id"
    count: Optional[int] = None                  # Count (count mode)

    TYPE = "node_query_result"  # Wire-dict type tag

    def to_dict(self) -> Dict[str, Any]:  # Tagged wire dict
        """Serialize to the wire dict form."""
        return {
            "type": self.TYPE,
            "nodes": [n.to_dict() for n in self.nodes] if self.nodes is not None else None,
            "rows": self.rows,
            "count": self.count,
        }

    @classmethod
    def from_dict(
        cls,
        d: Dict[str, Any]  # Tagged wire dict
    ) -> "NodeQueryResult":  # Reconstructed result
        """Reconstruct from the wire dict form (nested nodes via `GraphNode.from_dict`)."""
        nodes = d.get("nodes")
        return cls(
            nodes=[GraphNode.from_dict(n) for n in nodes] if nodes is not None else None,
            rows=d.get("rows"),
            count=d.get("count"),
        )

In [ ]:
#| export
@wire_type("graph.edge_query_result")
@dataclass
class EdgeQueryResult:
    """Typed result of an `EdgeQuery` — exactly one field is populated,
    mirroring the query's mode (default → `edges`, project → `rows`,
    count → `count`)."""
    edges: Optional[List[GraphEdge]] = None      # Full edges (default mode)
    rows: Optional[List[Dict[str, Any]]] = None  # Projected rows; always carry "id", "source_id", "target_id"
    count: Optional[int] = None                  # Count (count mode)

    TYPE = "edge_query_result"  # Wire-dict type tag

    def to_dict(self) -> Dict[str, Any]:  # Tagged wire dict
        """Serialize to the wire dict form."""
        return {
            "type": self.TYPE,
            "edges": [e.to_dict() for e in self.edges] if self.edges is not None else None,
            "rows": self.rows,
            "count": self.count,
        }

    @classmethod
    def from_dict(
        cls,
        d: Dict[str, Any]  # Tagged wire dict
    ) -> "EdgeQueryResult":  # Reconstructed result
        """Reconstruct from the wire dict form (nested edges via `GraphEdge.from_dict`)."""
        edges = d.get("edges")
        return cls(
            edges=[GraphEdge.from_dict(e) for e in edges] if edges is not None else None,
            rows=d.get("rows"),
            count=d.get("count"),
        )

In [ ]:
#| export
@wire_type("graph.raw_query_result")
@dataclass
class RawQueryResult:
    """Typed result of a `RawQuery` — tabular, backend-shaped (the columns are
    whatever the raw text selected; non-portable like the query itself)."""
    columns: List[str] = field(default_factory=list)      # Column names from the raw read
    rows: List[List[Any]] = field(default_factory=list)   # Result rows (positional, matching columns)
    row_count: int = 0                                    # len(rows) convenience
    backend: str = ""                                     # Backend that executed it (echo of RawQuery.backend)

    TYPE = "raw_query_result"  # Wire-dict type tag

    def to_dict(self) -> Dict[str, Any]:  # Tagged wire dict
        """Serialize to the wire dict form."""
        return {"type": self.TYPE, "columns": self.columns, "rows": self.rows,
                "row_count": self.row_count, "backend": self.backend}

    @classmethod
    def from_dict(
        cls,
        d: Dict[str, Any]  # Tagged wire dict
    ) -> "RawQueryResult":  # Reconstructed result
        """Reconstruct from the wire dict form."""
        return cls(columns=d.get("columns", []), rows=d.get("rows", []),
                   row_count=d.get("row_count", 0), backend=d.get("backend", ""))

In [ ]:
#| export
RESULT_TYPES: Dict[str, type] = {  # Tagged-dict registry for result dispatch (mirror of QUERY_TYPES)
    NodeQueryResult.TYPE: NodeQueryResult,
    EdgeQueryResult.TYPE: EdgeQueryResult,
    RawQueryResult.TYPE: RawQueryResult,
}


def result_from_dict(
    d: Dict[str, Any]  # Tagged wire dict ("type" in RESULT_TYPES)
) -> Any:  # NodeQueryResult | EdgeQueryResult | RawQueryResult
    """Reconstruct a query result from its tagged wire dict."""
    t = d.get("type")
    cls = RESULT_TYPES.get(t)
    if cls is None:
        raise CapabilityInputError(
            f"Unknown result type {t!r} (known: {sorted(RESULT_TYPES)})",
            fields_invalid=["type"],
        )
    return cls.from_dict(d)

In [ ]:
# The evidenced core read patterns, expressed typed (documentation-as-test):

# 1. C2/C3 spine read: Segments PART_OF a document, ordered by index, projected
#    (rows will carry "id" structurally; "sources" rides as a projectable pseudo-field)
spine = NodeQuery(
    label="Segment",
    related=RelationPredicate(relation_type="PART_OF", direction="out", node_id="doc-1"),
    order_by=OrderBy(prop="index"),
    project=["index", "text", "start_time", "end_time", "sources"],
)
assert query_from_dict(spine.to_dict()) == spine

# 2. Prune-candidate read: empty-or-null text Segments (the OR case = two
#    server-side-filtered queries unioned client-side; eq half shown)
empties = NodeQuery(
    label="Segment",
    where=[PropertyPredicate(prop="text", op="eq", value="")],
    related=RelationPredicate(relation_type="PART_OF", node_id="doc-1"),
    count=True,
)
assert query_from_dict(empties.to_dict()) == empties

# 3. Cross-transcript correction-cache read (stage-4 promotion): Corrections
#    whose CORRECTS target carries this content hash — ONE typed query via the
#    far-end node_source constraint (was a raw two-table JOIN, site C-6)
by_hash = NodeQuery(
    label="Correction",
    related=RelationPredicate(
        relation_type="CORRECTS", direction="out",
        node_source=SourcePredicate(content_hash="sha256:ab"),
    ),
)
assert query_from_dict(by_hash.to_dict()) == by_hash

# 4. C17 batch far end: active corrections for THESE segments in one round-trip
batch_corrections = NodeQuery(
    label="Correction",
    related=RelationPredicate(relation_type="CORRECTS", node_ids=["s1", "s2", "s3"]),
)
assert query_from_dict(batch_corrections.to_dict()) == batch_corrections

# 5. D13 verify aggregate (stage-4 promotion): count NEXT edges whose source
#    node is PART_OF the document — server-side, no segment-id materialization
next_count = EdgeQuery(
    relation_type="NEXT",
    source_related=RelationPredicate(relation_type="PART_OF", node_id="doc-1"),
    count=True,
)
assert query_from_dict(next_count.to_dict()) == next_count

# 6. Superseded-set read (stage-4 promotion): SUPERSEDES edges into an id set
superseded = EdgeQuery(relation_type="SUPERSEDES", target_ids=["c1", "c2"],
                       project=["target_id"])
assert query_from_dict(superseded.to_dict()) == superseded

# 7. Dotted property path (stage-4 promotion): document-scoped corrections
doc_corrections = NodeQuery(
    label="Correction",
    where=[PropertyPredicate(prop="payload.document_id", op="eq", value="doc-1")],
)
assert query_from_dict(doc_corrections.to_dict()) == doc_corrections

# batch-by-id (C17)
batch = NodeQuery(ids=["n1", "n2", "n3"])
assert query_from_dict(batch.to_dict()) == batch

# raw escape is marked: backend required
raw = RawQuery(text="SELECT COUNT(*) FROM nodes", backend="sqlite")
assert query_from_dict(raw.to_dict()) == raw
try:
    RawQuery(text="SELECT 1", backend="")
    raise AssertionError("should have raised")
except CapabilityInputError:
    pass

# validation: unknown op / bad direction / valueless null-op / conflicting endpoint forms
from cjm_substrate.core.errors import CapabilityInputError as PIE
for bad in (lambda: PropertyPredicate("text", "like", "%x%"),
            lambda: PropertyPredicate("text", "is_null", value=1),
            lambda: RelationPredicate("NEXT", direction="up"),
            lambda: RelationPredicate("NEXT", node_id="a", node_ids=["b"]),
            lambda: EdgeQuery(source_id="a", source_ids=["b"]),
            lambda: EdgeQuery(target_id="a", target_ids=["b"]),
            lambda: SourcePredicate()):
    try:
        bad()
        raise AssertionError("should have raised")
    except PIE:
        pass

# unknown query type raises
try:
    query_from_dict({"type": "graph_query"})
    raise AssertionError("should have raised")
except PIE:
    pass

In [ ]:
# Result DTO round-trips: tagged dict + wire envelope (stage 4)
import json as _json
from cjm_substrate.core.wire import wire_encode, wire_decode, WIRE_KIND_KEY

_node = GraphNode(id="n1", label="Segment", properties={"index": 0, "text": "hi"})
_edge = GraphEdge(id="e1", source_id="n1", target_id="n2", relation_type="NEXT")

nodes_res = NodeQueryResult(nodes=[_node])
rows_res = NodeQueryResult(rows=[{"id": "n1", "index": 0, "text": "hi"}])
count_res = NodeQueryResult(count=3579)
edge_res = EdgeQueryResult(edges=[_edge])
edge_rows = EdgeQueryResult(rows=[{"id": "e1", "source_id": "n1", "target_id": "n2", "decision": "corrected"}])
raw_res = RawQueryResult(columns=["c"], rows=[[1]], row_count=1, backend="sqlite")

# tagged-dict round-trip via the registry
for r in (nodes_res, rows_res, count_res, edge_res, edge_rows, raw_res):
    assert result_from_dict(r.to_dict()) == r

# wire-envelope round-trip (typed across the worker boundary, surviving JSON)
for r, kind in ((nodes_res, "graph.node_query_result"),
                (edge_res, "graph.edge_query_result"),
                (raw_res, "graph.raw_query_result")):
    env = wire_encode(r)
    assert env[WIRE_KIND_KEY] == kind
    assert wire_decode(_json.loads(_json.dumps(env))) == r, kind

# nested nodes reconstruct typed (sources list intact through the envelope)
decoded = wire_decode(_json.loads(_json.dumps(wire_encode(nodes_res))))
assert isinstance(decoded.nodes[0], GraphNode) and decoded.nodes[0].properties["index"] == 0

# unknown result type raises
try:
    result_from_dict({"type": "mystery_result"})
    raise AssertionError("should have raised")
except PIE:
    pass